# 02 — Ablation Study

<strong>SkillsAgent</strong> (`lib/skill_agent`) is the orchestrator for every condition. The harness passes three booleans (with_skills, with_docs, with_tools). None True => zero_shot; all True => with_skills.

<strong>Flow:</strong> Set up path → load benchmark + hydrate → build samples → create runner → run the 3 cells (or the conditions you choose). `AblationRunner(repo_root=repo_root)` defaults to the skill agent and resolves paths (results, prompts, skills) from `repo_root`.

<strong>Existing results:</strong> When merging, the runner loads existing from that local file first; if missing, it tries the **HuggingFace results repo**. If both are empty or missing, existing = 0.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EdyVision/pii-codex-skills-ablation-study/blob/main/notebooks/02_ablation_study.ipynb)

In [1]:
# Add lib so ablation_harness is importable; repo_root for Colab or local notebooks
import json
import sys
import random
import yaml
from pathlib import Path

from datasets import load_dataset

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root / "lib") not in sys.path:
    sys.path.insert(0, str(repo_root / "lib"))

from ablation_harness import AblationRunner

print(f"repo_root: {repo_root}")

repo_root: /Users/erosado/work/pii-codex-skills-ablation-study


### Load benchmark

Load the benchmark test split (sample indices + source); hydration fills in text and ground truth from source datasets next.

In [2]:
# Benchmark repo from repo root config.yaml (single source of truth)
with open(repo_root / "config.yaml") as f:
    _study_config = yaml.safe_load(f)
BENCHMARK_REPO = _study_config["compiled_dataset"]["repo_id"]

print(f"Loading benchmark from: {BENCHMARK_REPO}")
dataset = load_dataset(BENCHMARK_REPO, split="test")
print(f"Loaded {len(dataset)} samples")
print(f"Sources: {set(dataset['source'])}")

Loading benchmark from: EdyVision/pii-skills-ablation
Loaded 2000 samples
Sources: {'gretel_pii_masking', 'nvidia_nemotron_pii', 'ai4privacy'}


### Load source datasets

Load original HuggingFace datasets used to hydrate benchmark samples (text + ground truth).

In [3]:
print("Loading original datasets for hydration...")

ai4privacy = load_dataset("ai4privacy/pii-masking-300k", split="train")
print(f"  AI4Privacy: {len(ai4privacy):,} samples")

nvidia_ds = load_dataset("nvidia/Nemotron-PII", split="train")
print(f"  NVIDIA Nemotron-PII: {len(nvidia_ds):,} samples")

gretel_ds = load_dataset("gretelai/gretel-pii-masking-en-v1", split="train")
print(f"  Gretel PII masking: {len(gretel_ds):,} samples")

Loading original datasets for hydration...


Using the latest cached version of the dataset since ai4privacy/pii-masking-300k couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /Users/erosado/.cache/huggingface/datasets/ai4privacy___pii-masking-300k/default/0.0.0/3b99ae5980049a0dff660f424916afa9954e809b (last modified on Wed Feb 11 08:09:49 2026).


  AI4Privacy: 177,677 samples
  NVIDIA Nemotron-PII: 100,000 samples
  Gretel PII masking: 50,000 samples


### Hydrate samples

Fetch text and ground truth from source datasets using benchmark indices; filter out any samples that failed to hydrate.

In [4]:
if "dataset" not in globals():
    raise NameError("Run the 'Load benchmark' cell above first — it defines dataset.")
if (
    "ai4privacy" not in globals()
    or "nvidia_ds" not in globals()
    or "gretel_ds" not in globals()
):
    raise NameError("Run the 'Load source datasets' cell above first.")


def hydrate_sample(sample):
    """Fetch text and ground_truth from original dataset."""
    source = sample["source"]
    idx = sample["original_index"]

    text = None
    ground_truth = []

    if source == "ai4privacy":
        row = ai4privacy[idx]
        text = row["source_text"]
        if row.get("privacy_mask"):
            for item in row["privacy_mask"]:
                if isinstance(item, dict):
                    ground_truth.append(
                        {
                            "type": item.get("label", "UNK"),
                            "text": item.get("value", ""),
                            "start": item.get("start", 0),
                            "end": item.get("end", 0),
                        }
                    )

    elif source == "nvidia_nemotron_pii":
        row = nvidia_ds[idx]
        text = row.get("text", "")
        spans = row.get("spans")
        if spans is not None:
            if isinstance(spans, str):
                try:
                    spans = json.loads(spans)
                except (json.JSONDecodeError, TypeError):
                    spans = []
            for s in spans if isinstance(spans, list) else []:
                if isinstance(s, dict):
                    ground_truth.append(
                        {
                            "type": s.get("label", "UNK"),
                            "text": s.get("text", ""),
                            "start": s.get("start", 0),
                            "end": s.get("end", 0),
                        }
                    )

    elif source == "gretel_pii_masking":
        row = gretel_ds[idx]
        text = row.get("text", "")
        for e in row.get("entities") or []:
            if isinstance(e, dict):
                entity_text = e.get("entity", "")
                for t in e.get("types") or []:
                    ground_truth.append(
                        {"type": t, "text": entity_text, "start": 0, "end": 0}
                    )

    sample["text"] = text
    sample["ground_truth"] = json.dumps(ground_truth)
    return sample


print("Hydrating samples...")
dataset = dataset.map(hydrate_sample)
hydrated = dataset.filter(lambda x: x["text"] is not None)
print(f"Hydrated {len(hydrated)} / {len(dataset)} samples")
dataset = hydrated

Hydrating samples...
Hydrated 2000 / 2000 samples


In [5]:
# Set run type in the notebook: pilot = limit to pilot_n (200), main = full benchmark
RUN_TYPE = "main"  # "pilot" = n=200, "main" = full benchmark

with open(repo_root / "config.yaml") as f:
    study_config = yaml.safe_load(f)

experiment = study_config.get("experiment", {})
sampling = experiment.get("sampling", {})
pilot_n = sampling.get("pilot_n", 200)
seed = experiment.get("seed", 42)
max_samples = None if RUN_TYPE == "main" else pilot_n


def _text_length_bin(text):
    if not text:
        return "short"
    n = len(text)
    return "short" if n < 500 else "medium" if n < 2000 else "long"


def _pii_sig(gt_val):
    try:
        gt = json.loads(gt_val) if isinstance(gt_val, str) else (gt_val or [])
    except (json.JSONDecodeError, TypeError):
        gt = []
    if not gt:
        return "none"
    types = sorted(
        set(item.get("type", "UNK") for item in gt if isinstance(item, dict))
    )
    return "|".join(types[:3]) if types else "none"


def stratified_subsample(ds, n, seed_):
    N = len(ds)
    if n >= N:
        return ds
    strata = {}
    for idx in range(N):
        row = ds[idx]
        key = (
            row.get("source", "unknown"),
            _text_length_bin(row.get("text") or ""),
            _pii_sig(row.get("ground_truth")),
        )
        strata.setdefault(key, []).append(idx)
    random.seed(seed_)
    selected = []
    for indices in strata.values():
        share = max(1, min(int(n * len(indices) / N), len(indices)))
        random.shuffle(indices)
        selected.extend(indices[:share])
    if len(selected) > n:
        random.shuffle(selected)
        selected = selected[:n]
    elif len(selected) < n:
        remaining = [i for i in range(N) if i not in selected]
        random.shuffle(remaining)
        selected.extend(remaining[: n - len(selected)])
    random.shuffle(selected)
    return ds.select(selected)


if max_samples is None or max_samples >= len(dataset):
    dataset_for_run = dataset
    print(f"Config: run_type={RUN_TYPE!r}, no limit (full benchmark).")
else:
    dataset_for_run = stratified_subsample(dataset, max_samples, seed)
    print(
        f"Config: run_type={RUN_TYPE!r}, experiment.sampling.pilot_n={pilot_n}, seed={seed}."
    )
print(f"Running on {len(dataset_for_run)} samples (of {len(dataset)} hydrated).")

# Build samples list for harness (each needs "id")
samples = [dict(x) for x in dataset_for_run]
for i, s in enumerate(samples):
    s.setdefault("id", s.get("sample_id", s.get("id", str(i))))

Config: run_type='main', no limit (full benchmark).
Running on 2000 samples (of 2000 hydrated).


### Run Experiments with Ablation Runner

In [6]:
runner = AblationRunner(repo_root=repo_root)
# Use same run_type as sampling (main vs pilot) so Hub split and merge match
runner.config.run_type = RUN_TYPE
print(
    f"Runner will process {len(samples)} samples (run_type={RUN_TYPE}, see sampling cell above)."
)
results_dir = runner.config.results_dir
results_json = results_dir / "experiment_results.json"
print(f"Results dir (relative): {results_dir}")
print(f"Results dir (absolute): {results_dir.resolve()}")
print(f"Results JSON will be: {results_json.resolve()}")
print(
    "In Colab Files (left): open the folder tree to this path to see results after the run."
)

Runner will process 2000 samples (run_type=main, see sampling cell above).
Results dir (relative): /Users/erosado/work/pii-codex-skills-ablation-study/results
Results dir (absolute): /Users/erosado/work/pii-codex-skills-ablation-study/results
Results JSON will be: /Users/erosado/work/pii-codex-skills-ablation-study/results/experiment_results.json
In Colab Files (left): open the folder tree to this path to see results after the run.


#### Clear MLX / GPU memory (after interrupt or between runs as needed)

In [7]:
# import gc

# gc.collect()
# try:
#     import mlx.core as mx
#     mx.synchronize()
#     print("MLX synchronized; gc.collect() run.")
# except Exception:
#     print("gc.collect() run (MLX not used or not available).")

#### Split Runs

The harness runs only the conditions you pass; with_skills_runner gets (with_skills, with_docs, with_tools) so the same agent runs every condition.

In [8]:
# Multi-turn: with_tools
all_results = runner.run(samples, conditions=["with_tools"])

run_experiments: run_type='main' (Hub split will be main_v1)
Merging with 16000 existing results (re-running ['with_tools'] for ['llama3_8b', 'mistral_7b', 'qwen2_7b', 'gemma2_9b']).

Model: llama3_8b
Loading MLX model: mlx-community/Meta-Llama-3.1-8B-Instruct-4bit


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Model loaded successfully


llama3_8b:   0%|          | 0/2000 [00:00<?, ?sample/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/866 [00:00<?, ?B/s]

  Pushed to Hub: EdyVision/pii-skills-ablation-results (split=main_v1, n=24000)


In [8]:
# Single-turns: zero_shot and with_docs
all_results = runner.run(samples, conditions=["zero_shot", "with_docs"])

run_experiments: run_type='main' (Hub split will be main_v1)
  Split main_v1 not on Hub yet; no existing results to merge (first push will create it).

Model: llama3_8b
Loading MLX model: mlx-community/Meta-Llama-3.1-8B-Instruct-4bit


Model loaded successfully


  Push to Hub skipped: Features of the new split don't match the features of the existing splits on the hub: {'sample_id': Value('string'), 'source': Value('string'), 'model': Value('string'), 'condition': Value('string'), 'prompt_version': Value('string'), 'run_type': Value('string'), 'predictions': Value('string'), 'scores': Value('null'), 'error': Value('null'), 'tool_executed': Value('bool'), 'skill_viewed': Value('bool'), 'elapsed_seconds': Value('float64'), 'conversation_turns': Value('int64')} != {'sample_id': Value('string'), 'source': Value('string'), 'model': Value('string'), 'condition': Value('string'), 'prompt_version': Value('string'), 'run_type': Value('string'), 'predictions': Value('string'), 'scores': Value('string'), 'error': Value('null'), 'tool_executed': Value('bool'), 'skill_viewed': Value('bool'), 'elapsed_seconds': Value('float64'), 'conversation_turns': Value('int64')}
  zero_shot: 2000 done, 0 errors, 5482.8s — saved & pushed


  Push to Hub skipped: Features of the new split don't match the features of the existing splits on the hub: {'sample_id': Value('string'), 'source': Value('string'), 'model': Value('string'), 'condition': Value('string'), 'prompt_version': Value('string'), 'run_type': Value('string'), 'predictions': Value('string'), 'scores': Value('null'), 'error': Value('null'), 'tool_executed': Value('bool'), 'skill_viewed': Value('bool'), 'elapsed_seconds': Value('float64'), 'conversation_turns': Value('int64')} != {'sample_id': Value('string'), 'source': Value('string'), 'model': Value('string'), 'condition': Value('string'), 'prompt_version': Value('string'), 'run_type': Value('string'), 'predictions': Value('string'), 'scores': Value('string'), 'error': Value('null'), 'tool_executed': Value('bool'), 'skill_viewed': Value('bool'), 'elapsed_seconds': Value('float64'), 'conversation_turns': Value('int64')}
  with_docs: 2000 done, 0 errors, 14882.5s — saved & pushed


  Push to Hub skipped: Features of the new split don't match the features of the existing splits on the hub: {'sample_id': Value('string'), 'source': Value('string'), 'model': Value('string'), 'condition': Value('string'), 'prompt_version': Value('string'), 'run_type': Value('string'), 'predictions': Value('string'), 'scores': Value('null'), 'error': Value('null'), 'tool_executed': Value('bool'), 'skill_viewed': Value('bool'), 'elapsed_seconds': Value('float64'), 'conversation_turns': Value('int64')} != {'sample_id': Value('string'), 'source': Value('string'), 'model': Value('string'), 'condition': Value('string'), 'prompt_version': Value('string'), 'run_type': Value('string'), 'predictions': Value('string'), 'scores': Value('string'), 'error': Value('null'), 'tool_executed': Value('bool'), 'skill_viewed': Value('bool'), 'elapsed_seconds': Value('float64'), 'conversation_turns': Value('int64')}
  Checkpoint: saved and pushed (4000 results so far)

Model: mistral_7b
Loading MLX model: m

...

  Push to Hub skipped: Features of the new split don't match the features of the existing splits on the hub: {'sample_id': Value('string'), 'source': Value('string'), 'model': Value('string'), 'condition': Value('string'), 'prompt_version': Value('string'), 'run_type': Value('string'), 'predictions': Value('string'), 'scores': Value('null'), 'error': Value('null'), 'tool_executed': Value('bool'), 'skill_viewed': Value('bool'), 'elapsed_seconds': Value('float64'), 'conversation_turns': Value('int64')} != {'sample_id': Value('string'), 'source': Value('string'), 'model': Value('string'), 'condition': Value('string'), 'prompt_version': Value('string'), 'run_type': Value('string'), 'predictions': Value('string'), 'scores': Value('string'), 'error': Value('null'), 'tool_executed': Value('bool'), 'skill_viewed': Value('bool'), 'elapsed_seconds': Value('float64'), 'conversation_turns': Value('int64')}
  zero_shot: 2000 done, 0 errors, 12951.4s — saved & pushed


  Push to Hub skipped: Features of the new split don't match the features of the existing splits on the hub: {'sample_id': Value('string'), 'source': Value('string'), 'model': Value('string'), 'condition': Value('string'), 'prompt_version': Value('string'), 'run_type': Value('string'), 'predictions': Value('string'), 'scores': Value('null'), 'error': Value('null'), 'tool_executed': Value('bool'), 'skill_viewed': Value('bool'), 'elapsed_seconds': Value('float64'), 'conversation_turns': Value('int64')} != {'sample_id': Value('string'), 'source': Value('string'), 'model': Value('string'), 'condition': Value('string'), 'prompt_version': Value('string'), 'run_type': Value('string'), 'predictions': Value('string'), 'scores': Value('string'), 'error': Value('null'), 'tool_executed': Value('bool'), 'skill_viewed': Value('bool'), 'elapsed_seconds': Value('float64'), 'conversation_turns': Value('int64')}
  with_docs: 2000 done, 0 errors, 17828.6s — saved & pushed


  Push to Hub skipped: Features of the new split don't match the features of the existing splits on the hub: {'sample_id': Value('string'), 'source': Value('string'), 'model': Value('string'), 'condition': Value('string'), 'prompt_version': Value('string'), 'run_type': Value('string'), 'predictions': Value('string'), 'scores': Value('null'), 'error': Value('null'), 'tool_executed': Value('bool'), 'skill_viewed': Value('bool'), 'elapsed_seconds': Value('float64'), 'conversation_turns': Value('int64')} != {'sample_id': Value('string'), 'source': Value('string'), 'model': Value('string'), 'condition': Value('string'), 'prompt_version': Value('string'), 'run_type': Value('string'), 'predictions': Value('string'), 'scores': Value('string'), 'error': Value('null'), 'tool_executed': Value('bool'), 'skill_viewed': Value('bool'), 'elapsed_seconds': Value('float64'), 'conversation_turns': Value('int64')}
  Checkpoint: saved and pushed (16000 results so far)


  Push to Hub skipped: Features of the new split don't match the features of the existing splits on the hub: {'sample_id': Value('string'), 'source': Value('string'), 'model': Value('string'), 'condition': Value('string'), 'prompt_version': Value('string'), 'run_type': Value('string'), 'predictions': Value('string'), 'scores': Value('null'), 'error': Value('null'), 'tool_executed': Value('bool'), 'skill_viewed': Value('bool'), 'elapsed_seconds': Value('float64'), 'conversation_turns': Value('int64')} != {'sample_id': Value('string'), 'source': Value('string'), 'model': Value('string'), 'condition': Value('string'), 'prompt_version': Value('string'), 'run_type': Value('string'), 'predictions': Value('string'), 'scores': Value('string'), 'error': Value('null'), 'tool_executed': Value('bool'), 'skill_viewed': Value('bool'), 'elapsed_seconds': Value('float64'), 'conversation_turns': Value('int64')}

Results saved to /Users/erosado/work/pii-codex-skills-ablation-study/results/experiment_res

  Pushed to Hub: EdyVision/pii-skills-ablation-results (split=main_v1, n=16000)


In [8]:
# Multi-turn: with_skills
all_results = runner.run(samples, conditions=["with_skills"])

run_experiments: run_type='main' (Hub split will be main_v1)
Merging with 24000 existing results (re-running ['with_skills'] for ['llama3_8b', 'mistral_7b', 'qwen2_7b', 'gemma2_9b']).

Model: llama3_8b
Loading MLX model: mlx-community/Meta-Llama-3.1-8B-Instruct-4bit


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Model loaded successfully


llama3_8b:   0%|          | 0/2000 [00:00<?, ?sample/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/866 [00:00<?, ?B/s]

  Pushed to Hub: EdyVision/pii-skills-ablation-results (split=main_v1, n=32000)


#### Results

In [9]:
import pandas as pd

# Build a flat table from all_results for summary
rows = []
for r in all_results:
    gt = r.get("ground_truth")
    if isinstance(gt, str):
        try:
            gt = json.loads(gt)
        except (json.JSONDecodeError, TypeError):
            gt = []
    n_gt = len(gt) if isinstance(gt, list) else 0
    preds = r.get("predictions", [])
    rows.append(
        {
            "model": r.get("model"),
            "condition": r.get("condition"),
            "source": r.get("source"),
            "sample_id": r.get("sample_id"),
            "n_predictions": len(preds),
            "n_ground_truth": n_gt,
            "has_error": r.get("error") is not None,
            "tool_executed": r.get("tool_executed", False),
            "skill_viewed": r.get("skill_viewed", False),
            "elapsed_seconds": r.get("elapsed_seconds"),
        }
    )
df_results = pd.DataFrame(rows)

# Summary by model x condition
# All three behavioral metrics: tool_calls, docs_viewed, skills_viewed (docs/skills both from view_skill)
summary = (
    df_results.groupby(["model", "condition"])
    .agg(
        n=("sample_id", "count"),
        errors=("has_error", "sum"),
        mean_preds=("n_predictions", "mean"),
        mean_gt=("n_ground_truth", "mean"),
        tool_calls=("tool_executed", "sum"),
        skills_viewed=("skill_viewed", "sum"),
    )
    .round(2)
    .reset_index()
)
print(f"Total results: {len(all_results)}")
display(summary)
print("\nPer-row describe (n_predictions, n_ground_truth):")
display(df_results[["n_predictions", "n_ground_truth"]].describe())

Total results: 32000


,model,condition,n,errors,mean_preds,mean_gt,tool_calls,skills_viewed
0,gemma2_9b,with_docs,2000,0,3.88,5.68,0,0
1,gemma2_9b,with_skills,2000,0,5.15,5.68,2000,2000
2,gemma2_9b,with_tools,2000,0,5.15,5.68,2000,0
3,gemma2_9b,zero_shot,2000,0,3.70,5.68,0,0
4,llama3_8b,with_docs,2000,0,3.78,5.68,0,0
5,llama3_8b,with_skills,2000,0,5.42,5.68,2000,2000
6,llama3_8b,with_tools,2000,0,5.48,5.68,2000,0
7,llama3_8b,zero_shot,2000,0,4.94,5.68,0,0
8,mistral_7b,with_docs,2000,0,3.73,5.68,0,0
9,mistral_7b,with_skills,2000,0,4.46,5.68,1999,948



Per-row describe (n_predictions, n_ground_truth):


,n_predictions,n_ground_truth
count,32000.000000,32000.00000
mean,4.478906,5.68500
std,3.363430,3.95326
min,0.000000,1.00000
25%,2.000000,3.00000
50%,4.000000,5.00000
75%,6.000000,7.00000
max,33.000000,46.00000


Result saved to `runner.config.results_dir / experiment_results.json` (e.g. `results/experiment_results.json`). Results are **incrementally pushed to Hugging Face** after each condition (same format as before), so notebook 03 can pull them and analyze without re-running. After scoring, the runner saves and pushes again so the Hub has scores.

Next: notebook 03 for analyses and figures.